# Effect of Force-Level Conditioning

This notebook compares image-only and force-level-conditioned object and object-region classification across training seeds `156`, `157`, and `158`.

Comparisons use the internal `test` and `test_unseen_matched` splits. `validation_unseen_matched` is included in the CSV table for reference and excluded from the plot because it is not an independent test result. The related unseen split (`test_unseen_related`) is excluded for object and object-region classification because these results are not scored.

Accuracy differences are percentage points; macro F1 differences retain the original 0–1 scale in the CSV.

It exports one CSV table of individual seed differences and summary statistics, plus one plot of mean differences with sample-standard-deviation error bars.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Match the main repository's typography
sns.set_theme(
    style="darkgrid",
    palette="deep",
    font="DejaVu Sans",
    rc={"font.size": 11, "legend.title_fontsize": 12},
)

RESULTS_DIR = Path("../results/final_evaluation")
OUTPUT_DIR = Path("../exports/conditioning_comparison")
SEEDS = [156, 157, 158]
LABELS = ["object", "object_region"]

# Only independent test results appear in the plot
PLOT_METRIC_COLUMNS = [
    "test_accuracy",
    "test_macro_f1_avg",
    "test_unseen_matched_accuracy",
    "test_unseen_matched_macro_f1_avg",
]

# Include the matched unseen validation split in the CSV table only
CONDITIONING_METRIC_COLUMNS = PLOT_METRIC_COLUMNS + [
    "validation_unseen_matched_accuracy",
    "validation_unseen_matched_macro_f1_avg",
]

In [ ]:
def load_conditioning_results(label):
    """
    Load image-only and conditioned results matched by training seed.

    Args:
        label (str): The classification label to compare.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: Image-only and conditioned result rows,
            indexed in training-seed order.
    """

    results_by_model = []
    for task in [f"{label}_classification", f"{label}_conditioned_classification"]:
        results_path = RESULTS_DIR / task / "experiments.parquet"
        results = pd.read_parquet(results_path)
        results = results.loc[results["seed"].isin(SEEDS)].copy()
        results = results.set_index("seed").loc[SEEDS, CONDITIONING_METRIC_COLUMNS]
        results_by_model.append(results)

    return results_by_model[0], results_by_model[1]


def calculate_conditioning_deltas(main, conditioned):
    """
    Calculate conditioned-minus-image-only differences within each training seed.

    Args:
        main (pd.DataFrame): The image-only results indexed by training seed.
        conditioned (pd.DataFrame): The conditioned results indexed by training seed.

    Returns:
        pd.DataFrame: Per-seed differences for each comparison metric.
    """

    return (
        conditioned.loc[SEEDS, CONDITIONING_METRIC_COLUMNS]
        - main.loc[SEEDS, CONDITIONING_METRIC_COLUMNS]
    )


def export_conditioning_comparison():
    """
    Export per-seed differences and their summary statistics for both labels.

    Returns:
        pd.DataFrame: One row per label and metric, with paired seed differences
            and their summary statistics.
    """

    rows = []
    for label in LABELS:
        main, conditioned = load_conditioning_results(label)
        deltas = calculate_conditioning_deltas(main, conditioned)

        for metric in CONDITIONING_METRIC_COLUMNS:
            values = deltas[metric]
            row = {"label": label, "metric": metric}
            row.update({f"seed_{seed}_delta": values.loc[seed] for seed in SEEDS})
            row.update(
                {
                    "n": values.count(),
                    "mean_delta": values.mean(),
                    "std_delta": values.std(ddof=1),
                    "min_delta": values.min(),
                    "max_delta": values.max(),
                }
            )
            rows.append(row)

    comparison = pd.DataFrame(rows)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    comparison.to_csv(OUTPUT_DIR / "conditioning_comparison.csv", index=False)
    return comparison


In [ ]:
def plot_conditioning_comparison(comparison):
    """
    Plot test-split mean paired differences with sample-standard-deviation error bars.

    Args:
        comparison (pd.DataFrame): Per-seed differences and summary statistics.
    """

    # Plot only the test splits, keeping the original metric and label order
    plot_data = comparison.pivot(
        index="metric", columns="label", values="mean_delta"
    ).reindex(index=PLOT_METRIC_COLUMNS, columns=LABELS)
    plot_errors = comparison.pivot(
        index="metric", columns="label", values="std_delta"
    ).reindex(index=PLOT_METRIC_COLUMNS, columns=LABELS)

    # Scale both macro F1 means and standard deviations to percentage points
    f1_metrics = [metric for metric in PLOT_METRIC_COLUMNS if "f1" in metric]
    plot_data.loc[f1_metrics] *= 100
    plot_errors.loc[f1_metrics] *= 100

    metric_names = {
        "test_accuracy": "Internal Test Accuracy",
        "test_macro_f1_avg": "Internal Test Macro F1",
        "test_unseen_matched_accuracy": "Unseen Matched Accuracy",
        "test_unseen_matched_macro_f1_avg": "Unseen Matched Macro F1",
    }
    label_names = {"object": "Object", "object_region": "Object Region"}
    plot_data = plot_data.rename(index=metric_names, columns=label_names)
    plot_errors = plot_errors.rename(index=metric_names, columns=label_names)

    # Use the shared theme and show one sample SD of the paired differences
    fig, ax = plt.subplots(figsize=(14, 5))
    plot_data.plot.bar(
        ax=ax,
        yerr=plot_errors,
        capsize=4,
        color=sns.color_palette(n_colors=len(LABELS)),
    )

    # Zero marks no change in performance from adding force-level conditioning
    ax.axhline(
        0,
        color="black",
        linestyle="--",
        linewidth=0.8,
    )
    ax.set_ylabel("Conditioned − Unconditioned (% Points)")
    ax.set_xlabel("")
    ax.legend(title="Label")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    fig.tight_layout()

    # Save the single comparison plot
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        OUTPUT_DIR / "conditioning_comparison.png",
        dpi=300,
        bbox_inches="tight",
    )


In [ ]:
# Export the full table, including matched unseen validation results
conditioning_comparison = export_conditioning_comparison()
conditioning_comparison

In [ ]:
# Plot the test metrics only (validation rows remain in the CSV)
plot_conditioning_comparison(conditioning_comparison)